In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.nn.init as init
import scipy.linalg as sci
import scipy.io as sio
import numpy as np
import matplotlib.pyplot as plt

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [7]:
## System parameters
M = 64 #Number of BS antennas
P = 1 #Power
K =  2 #Number of users
L = 8 #Number of pilots
B = 30 #Number of feedback bits per user

## Limited scattering channel parameters
LSF_UE = np.array([0.0,0.0],dtype=np.float32) #Mean of path gains for K users
Mainlobe_UE= np.array([0,0],dtype=np.float32) #Center of the AoD range for K users
HalfBW_UE = np.array([30.0,30.0],dtype=np.float32) #Half of the AoD range for K users

# SNR
snr_dl = 10 #SNR in dB
noise_std_dl = np.float32(np.sqrt(1/2)*np.sqrt(P/10**(snr_dl/10))) #STD of the Gaussian noise (per real dim.)

In [ ]:
## Learning parameters
initial_run = 1 #1: starts training from scratch; 0: resumes training 
n_epochs = 5000 #Number of training epochs, for observing the current performance set it to 0
learning_rate = 0.0001 #Learning rate

batch_size = 1024 #Mini-batch size
test_size = 10000 #Size of the validation/test set
batch_per_epoch = 20 #Numbers of mini-batches per epoch

anneal_param = 0.5 #Initial annealing parmeter
annealing_rate = 1.001 #Annealing rate

### Pilot Sequence 초기화 -- DFT Matrix 이용

In [9]:
DFT_Matrix = sci.dft(M) 
X_init = DFT_Matrix[0::int(np.ceil(M/L)),:] 
Xp_init = np.sqrt(P/M)*X_init
Xp_r_init = torch.Tensor(np.float32(np.real(Xp_init))).to(device)
Xp_i_init = torch.Tensor(np.float32(np.imag(Xp_init))).to(device)
## Pilot sequence cuda 지정 완료

### Matrix 연산 함수

In [20]:
def mult_mod(M, N, left_right):
    tensor_shape = M.shape
    dims = N.shape 

    if left_right == 'r':
        # M: (batch_size, n, m) N: (m, p)
        n = tensor_shape[1]
        m = dims[0]
        p = dims[1]

        # PyTorch의 행렬 곱
        y = torch.reshape(torch.matmul(M.view(-1, m), N), (-1, n, p))

    elif left_right == 'l':
        # M: (batch_size, n, m)
        # N: (p, n)
        m = tensor_shape[2]
        p = dims[0]
        n = dims[1]

        # PyTorch에서는 `permute()`로 전치 가능
        MT = torch.Tensor(M).permute(0, 2, 1)  # (batch_size, m, n)
        NT = N.T  # (n, p)

        MTNT = torch.reshape(torch.matmul(MT.view(-1, n), NT), (-1, m, p))
        y = MTNT.permute(0, 2, 1)  # (batch_size, n, p)로 변환

    return y.to(device)

def mult_mod_complex(Mr, Mi, Nr, Ni, left_right):
    yr = mult_mod(Mr, Nr, left_right) - mult_mod(Mi, Ni, left_right)
    yi = mult_mod(Mr, Ni, left_right) + mult_mod(Mi, Nr, left_right)
    return yr.to(device), yi.to(device)

### Batch data 생성 함수

In [21]:
## 리스트 인덱싱은 [층, 행, 열], [행, 열]임!
def generate_batch_data(batch_size,M,K,
                        Lp,#number of paths
                        LSF_UE #Mean of path gains for K users
                        ,Mainlobe_UE #Center of the AoD range for K users
                        ,HalfBW_UE #Half of the AoD range for K users
                        ):
    alphaR_input = np.zeros((batch_size,Lp,K))
    alphaI_input = np.zeros((batch_size,Lp,K))
    theta_input = np.zeros((batch_size,Lp,K))
    for kk in range(K): # for the number of users
        alphaR_input[:,:,kk] = np.random.normal(loc=LSF_UE[kk], scale=1.0/np.sqrt(2), size=[batch_size,Lp])
        alphaI_input[:,:,kk] = np.random.normal(loc=LSF_UE[kk], scale=1.0/np.sqrt(2), size=[batch_size,Lp])
        theta_input[:,:,kk] = np.random.uniform(low=Mainlobe_UE[kk]-HalfBW_UE[kk], high=Mainlobe_UE[kk]+HalfBW_UE[kk], size=[batch_size,Lp])
 
    #### Actual Channel
    from0toM = np.float32(np.arange(0, M, 1))
    alpha_act = alphaR_input + 1j*alphaI_input
    theta_act = (np.pi/180)*theta_input
    
    h_act = np.complex128(np.zeros((batch_size,M,K)))
    hR_act = np.float32(np.zeros((batch_size,M,K)))
    hI_act = np.float32(np.zeros((batch_size,M,K)))
    
    for kk in range(K):
        for ll in range(Lp):
            theta_act_expanded_temp = np.tile(np.reshape(theta_act[:,ll,kk],[-1,1]),(1,M))
            response_temp = np.exp(1j*np.pi*np.multiply(np.sin(theta_act_expanded_temp),from0toM))
            alpha_temp = np.reshape(alpha_act[:,ll,kk],[-1,1])
            h_act[:,:,kk] += (1/np.sqrt(Lp))*alpha_temp*response_temp
        hR_act[:,:,kk] = np.real(h_act[:,:,kk])
        hI_act[:,:,kk] = np.imag(h_act[:,:,kk])
        
    h_act = torch.tensor(h_act, dtype=torch.complex64).to(device)
    hR_act = torch.tensor(hR_act, dtype=torch.float32).to(device)
    hI_act = torch.tensor(hI_act, dtype=torch.float32).to(device)
        
    return(h_act, hR_act, hI_act)

In [22]:
## 리스트 인덱싱은 [층, 행, 열], [행, 열]임!
def generate_varyingpath_batch_data(batch_size,M,K,
                        Lp,#number of paths : given by list
                        LSF_UE #Mean of path gains for K users
                        ,Mainlobe_UE #Center of the AoD range for K users
                        ,HalfBW_UE #Half of the AoD range for K users
                        ):
    for ll in Lp:
        h_act_temp, hR_act_temp, hI_act_temp = generate_batch_data(batch_size, M, K, ll, LSF_UE, Mainlobe_UE, HalfBW_UE)
        if Lp.index(ll) == 0:
            h_act = h_act_temp
            hR_act = hR_act_temp
            hI_act = hI_act_temp
        else:
            h_act = torch.cat((h_act, h_act_temp))
            hR_act = torch.cat((hR_act, hR_act_temp))
            hI_act = torch.cat((hI_act, hI_act_temp))
            
    return(h_act, hR_act, hI_act)

### Downlink Pilot Training -- Pilot Sequence as DNN parameters

In [23]:
## hR, hI 만 cuda로 들어가면 됨됨
class DLTrainingPhase(nn.Module):
    def __init__(self, P, noise_std_dl):
        super(DLTrainingPhase, self).__init__()
        
        # noise/annealing parameter
        self.noise_std = torch.tensor(noise_std_dl, dtype=torch.float32).to(device)
        self.aneal = torch.tensor(1.0, dtype=torch.float32).to(device)
        
        # Pilot sequence - Use pre-initialized values
        self.Xp_r = nn.Parameter(Xp_r_init.clone().to(device))
        self.Xp_i = nn.Parameter(Xp_i_init.clone().to(device))
        
        # Power normalizing
        self.P = P
        self.normalize_pilot()
        
    def normalize_pilot(self):
        # Function : Normalizing the pilot sequence vectors
        norm_X = torch.sqrt(torch.sum(self.Xp_r**2 + self.Xp_i**2, dim = 1, keepdim = True)) # (. , * , . ) *에 대해 sum 수행
        self.Xp_r.data = torch.sqrt(torch.tensor(self.P)) * (self.Xp_r / norm_X)   
        self.Xp_i.data = torch.sqrt(torch.tensor(self.P)) * (self.Xp_i / norm_X)
        
    def forward(self, hR, hI, K, M, L):
        y_nless = {}
        y_noisy = {}
        
        for kk in range(K):
            hR_temp = hR[:, :, kk].reshape(-1, M, 1) # 차원 (batch_size, M)
            hI_temp = -1 * hI[:, :, kk].reshape(-1, M, 1)

            # 복소수 행렬 곱 수행
            y_nless_r, y_nless_i = mult_mod_complex(hR_temp, hI_temp, self.Xp_r, self.Xp_i, 'l')

            # 실수 및 허수 결합 -> 복소수로 결합 아니고 real representation
            y_nless[kk] = torch.cat([y_nless_r.view(-1, L), y_nless_i.view(-1, L)], dim=1)

            # 가우시안 노이즈 추가
            noise = torch.randn_like(y_nless[kk]) * self.noise_std
            y_noisy[kk] = y_nless[kk] + noise
        
        return y_noisy

### UE side DNN - Quantizer for CSI feedback

In [24]:
class UE_DNN(nn.Module):
    def __init__(self, L, B, K, anneal = 1.0):
        super(UE_DNN, self).__init__()
        self.anneal = anneal
        self.input_dim = 2*L
        self.K=K
        
        self.model = nn.Sequential(
            nn.BatchNorm1d(self.input_dim),
            nn.Linear(self.input_dim, 1024),
            nn.ReLU(),
            
            nn.BatchNorm1d(1024),
            nn.Linear(1024, 512),
            nn.ReLU(),
            
            nn.BatchNorm1d(512),
            nn.Linear(512, 256),
            nn.ReLU(),
            
            nn.BatchNorm1d(256),
            nn.Linear(256, B)
        )
        self.model.to(device)
        
    def forward(self, x):
        InfoBits = {0:0}
        for kk in range(self.K):
            InfoBits_linear = self.model(x[kk])  # 신경망을 통과한 값

            # Straight-Through Estimator (STE) 적용
            InfoBits_tanh = torch.tanh(self.anneal * InfoBits_linear)
            InfoBits_sign = torch.sign(InfoBits_linear)

            # Forward: Sign 값을 사용, Backward: Tanh gradient 사용
            InfoBits[kk] = InfoBits_tanh + (InfoBits_sign - InfoBits_tanh).detach()
        
        return InfoBits # K * (batch, M)

In [25]:
class BS_DNN(nn.Module):
    def __init__(self, M, K, B, P):
        super(BS_DNN, self).__init__()
        self.M = M
        self.K = K
        self.B = B
        self.P = P
        
        self.model = nn.Sequential(
            nn.Linear(K * B, 1024),
            nn.ReLU(),
            nn.BatchNorm1d(1024),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, 2* M * K)  # Precoder 출력 (실수 및 허수 파트)
        )
        
        self.model.to(device)
    
    def forward(self, DNN_input_BS):
        # DNN_input_BS : UE에서 생성한 정보 입력 - K * (batch, M)
        # hR, hI : channel matrix - (batch_size, M, K) - real/imag part
        
        #batch_size = DNN_input_BS[0].shape()
        
        # 0. Precoder input 생성
        for kk in range(self.K):
            if kk == 0:
                precoder_input = DNN_input_BS[0]
            else:
                precoder_input = torch.concat((precoder_input, DNN_input_BS[kk]), dim = 1) # (batch, K, M)
        
        # 1. Generate the precoder
        precoder_output = self.model(precoder_input) # (batch_size, 2*M*K)
        
        # 2. seperate real/imag part
        V_r, V_i = torch.chunk(precoder_output, 2, dim=1)
        V_r.to(torch.float32)
        V_i.to(torch.float32)
        
        # 3. power normalization
        norm_V = torch.sqrt(torch.sum(V_r**2 + V_i**2, dim=1, keepdim=True))
        V_r = np.sqrt(self.P) * (V_r / norm_V)
        V_i = np.sqrt(self.P) * (V_i / norm_V)
        
        return V_r, V_i

### 모델 학습하기

In [26]:
## Loss function 정의
class CustomLoss(nn.Module):
    def __init__(self, K):
        super(CustomLoss, self).__init__()
        self.K = K
    
    def forward(self, V_r, V_i, h_act, noise_std): # V_r, V_i : (batch, M * K), h_act : (batch, M, K)
        v_r = V_r.reshape(-1, M, K)
        v_i = V_i.reshape(-1, M, K)
        V = torch.complex(v_r, v_i)
        hH_V = torch.bmm(h_act.mH, V)
        hH_V = torch.pow(abs(hH_V), 2)
        noise_std = torch.tensor(noise_std).to(device)
        
        rate = torch.zeros(h_act.shape[0]).to(device)
        for kk in range(self.K):
            sig_pow = torch.sum(hH_V[:, kk, :], dim=-1)
            wanted_sig = hH_V[:, kk, kk]
            rate += torch.log2(1 + ( wanted_sig / (sig_pow - wanted_sig + 2 * noise_std ** 2)) ).to(device)

        loss = torch.mean( -rate )
 
        return loss
    
Lossfunc = CustomLoss(K)

In [27]:
## Test set 생성
Lp = [2, 3, 4, 5, 6, 7, 8]
h_test, hR_test, hI_test = generate_varyingpath_batch_data(1429, M, K, Lp, LSF_UE, Mainlobe_UE, HalfBW_UE)
h_final = {}
hR_final = {}
hI_final = {}
for ll in Lp:
    h_final[ll], hR_final[ll], hI_final[ll] = generate_batch_data(10000, M, K, ll, LSF_UE, Mainlobe_UE, HalfBW_UE)

In [43]:
## Loop
experiment_result=[]

anneal_param = 0.5
pilot_train = DLTrainingPhase(P, noise_std_dl)
ue_dnn = UE_DNN(L, B, K, annealing_rate)
bs_dnn = BS_DNN(M, K, B, P)

## Optimizer 설정
optimizer = optim.Adam(list(pilot_train.parameters())+list(ue_dnn.parameters())+list(bs_dnn.parameters())
                    , lr=learning_rate)

best_loss = float('inf')
worse_count = 0 # early stopping을 위함

epoch = 0
while True:
    optimizer.zero_grad()
    
    for _ in range(batch_per_epoch):
        # tarin dataset 생성
        h_batch, hR_batch, hI_batch = generate_varyingpath_batch_data(146, M, K, Lp, LSF_UE, Mainlobe_UE, HalfBW_UE)
        
        y_pilot = pilot_train(hR_batch, hI_batch, K, M, L) # K user의 파일럿 수신 신호 리스트
        UE_Feedback = ue_dnn(y_pilot)
        V_r, V_i = bs_dnn(UE_Feedback)
        loss = Lossfunc(V_r, V_i, h_batch, noise_std_dl)
        
        loss.backward()
        optimizer.step()
        
    if epoch % 10==0:
        print(f'🚩 Lp in [2, 8] system, {epoch} epoch')
        y_pilot = pilot_train(hR_test, hI_test, K, M, L)
        UE_Feedback = ue_dnn(y_pilot)
        V_r_test, V_i_test = bs_dnn(UE_Feedback)
        loss_test = Lossfunc(V_r_test, V_i_test, h_test, noise_std_dl)
        
        if loss_test < best_loss: # 성능이 더 좋아졌다면
            worse_count = 0
            best_loss = loss_test
            
            print(f"✅ Model saved at epoch {epoch}, Loss: {best_loss:.5f}, Worse count reset to 0")
        else:
            worse_count += 10
            print(f"❌ No improvement for {worse_count} epochs")
            anneal_param = anneal_param*annealing_rate # annealing parameter 증가가
        
        print('epoch',epoch,' anneal_param:%4.4f'%anneal_param)
        print('         loss_test:%2.5f'%-loss_test,'  best_test:%2.5f'%-best_loss)
        
        if worse_count == 300:
            print("❗ Stopping experiment : No more improvement")
            break
    epoch += 1
for ll in Lp:   
    y_pilot_test_Final = pilot_train(hR_final[ll], hI_final[ll], K, M, L)
    UE_Feedback_test_Final = ue_dnn(y_pilot_test_Final)
    V_r_test_Final, V_i_test_Final = bs_dnn(UE_Feedback_test_Final)
    loss_test_Final = Lossfunc(V_r_test_Final, V_i_test_Final, h_final[ll], noise_std_dl)

    print(-loss_test_Final)
    experiment_result.append(-loss_test_Final.item())

🚩 Lp in [2, 8] system, 0 epoch
✅ Model saved at epoch 0, Loss: -2.07266, Worse count reset to 0
epoch 0  anneal_param:0.5000
         loss_test:2.07266   best_test:2.07266
🚩 Lp in [2, 8] system, 10 epoch
✅ Model saved at epoch 10, Loss: -5.46689, Worse count reset to 0
epoch 10  anneal_param:0.5000
         loss_test:5.46689   best_test:5.46689
🚩 Lp in [2, 8] system, 20 epoch
✅ Model saved at epoch 20, Loss: -6.42123, Worse count reset to 0
epoch 20  anneal_param:0.5000
         loss_test:6.42123   best_test:6.42123
🚩 Lp in [2, 8] system, 30 epoch
✅ Model saved at epoch 30, Loss: -6.79145, Worse count reset to 0
epoch 30  anneal_param:0.5000
         loss_test:6.79145   best_test:6.79145
🚩 Lp in [2, 8] system, 40 epoch
✅ Model saved at epoch 40, Loss: -7.01242, Worse count reset to 0
epoch 40  anneal_param:0.5000
         loss_test:7.01242   best_test:7.01242
🚩 Lp in [2, 8] system, 50 epoch
✅ Model saved at epoch 50, Loss: -7.13724, Worse count reset to 0
epoch 50  anneal_param:0.5000


In [47]:
import pickle
print(experiment_result)
with open('Proposed DNN trained Lp=[2, 8]', 'wb') as fw:
    pickle.dump(experiment_result, fw)

[12.912054061889648, 12.777671813964844, 12.457122802734375, 11.805867195129395, 11.019146919250488, 10.365612030029297, 9.84095573425293]


In [ ]:
'''
save_path = './Lp_trained_2to8_params.pth'
torch.save({
                    'pilot_train' : pilot_train.state_dict(),
                    'bs_dnn' : bs_dnn.state_dict(),
                    'ue_dnn' : ue_dnn.state_dict(),
                    'optimizer' : optimizer.state_dict(),
                    'best_loss' : best_loss
}, save_path)
'''

In [ ]:
'''
checkpoint = torch.load('./Lp_trained_2to8_params.pth')

# 각 모델에 저장된 state_dict를 로드
pilot_train.load_state_dict(checkpoint['pilot_train'])
bs_dnn.load_state_dict(checkpoint['bs_dnn'])
ue_dnn.load_state_dict(checkpoint['ue_dnn'])

with torch.no_grad():
    for ll in Lp:   
        y_pilot_test_Final = pilot_train(hR_final[ll], hI_final[ll], K, M, L)
        UE_Feedback_test_Final = ue_dnn(y_pilot_test_Final)
        V_r_test_Final, V_i_test_Final = bs_dnn(UE_Feedback_test_Final)
        loss_test_Final = Lossfunc(V_r_test_Final, V_i_test_Final, h_final[ll], noise_std_dl)

        print(-loss_test_Final.item())
        experiment_result.append(-loss_test_Final.item())
'''

C:\Users\unist\AppData\Local\Temp\ipykernel_16716\3156967262.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load('./Lp_trained_2to8_params.pth')


1.6540815830230713
1.8881360292434692
2.1246280670166016
2.2171738147735596
2.240689754486084
2.3306312561035156
2.344940185546875


In [51]:
## Test set 생성
Lp = [2, 3, 4, 5, 6, 7, 8]
h_test, hR_test, hI_test = generate_batch_data(10000, M, K, 2, LSF_UE, Mainlobe_UE, HalfBW_UE)
h_final = {}
hR_final = {}
hI_final = {}
for ll in Lp:
    h_final[ll], hR_final[ll], hI_final[ll] = generate_batch_data(10000, M, K, ll, LSF_UE, Mainlobe_UE, HalfBW_UE)

In [53]:
## Loop
experiment_result_Lp2=[]

anneal_param = 0.5
pilot_train = DLTrainingPhase(P, noise_std_dl)
ue_dnn = UE_DNN(L, B, K, annealing_rate)
bs_dnn = BS_DNN(M, K, B, P)

## Optimizer 설정
optimizer = optim.Adam(list(pilot_train.parameters())+list(ue_dnn.parameters())+list(bs_dnn.parameters())
                    , lr=learning_rate)

best_loss = float('inf')
worse_count = 0 # early stopping을 위함

epoch = 0
while True:
    optimizer.zero_grad()
    
    for _ in range(batch_per_epoch):
        # tarin dataset 생성
        h_batch, hR_batch, hI_batch = generate_batch_data(1024, M, K, 2, LSF_UE, Mainlobe_UE, HalfBW_UE)
        
        y_pilot = pilot_train(hR_batch, hI_batch, K, M, L) # K user의 파일럿 수신 신호 리스트
        UE_Feedback = ue_dnn(y_pilot)
        V_r, V_i = bs_dnn(UE_Feedback)
        loss = Lossfunc(V_r, V_i, h_batch, noise_std_dl)
        
        loss.backward()
        optimizer.step()
        
    if epoch % 10==0:
        print(f'🚩 Lp in 2 system, {epoch} epoch')
        y_pilot_test = pilot_train(hR_test, hI_test, K, M, L)
        UE_Feedback_test = ue_dnn(y_pilot_test)
        V_r_test, V_i_test = bs_dnn(UE_Feedback_test)
        loss_test = Lossfunc(V_r_test, V_i_test, h_test, noise_std_dl)
        
        if loss_test < best_loss: # 성능이 더 좋아졌다면
            worse_count = 0
            best_loss = loss_test
            
            print(f"✅ Model saved at epoch {epoch}, Loss: {best_loss:.5f}, Worse count reset to 0")
        else:
            worse_count += 10
            print(f"❌ No improvement for {worse_count} epochs")
            anneal_param = anneal_param*annealing_rate # annealing parameter 증가가
        
        print('epoch',epoch,' anneal_param:%4.4f'%anneal_param)
        print('         loss_test:%2.5f'%-loss_test,'  best_test:%2.5f'%-best_loss)
        
        if worse_count == 300:
            print("❗ Stopping experiment : No more improvement")
            break
    epoch += 1
    
for ll in Lp:   
    y_pilot_final = pilot_train(hR_final[ll], hI_final[ll], K, M, L)
    UE_Feedback_final = ue_dnn(y_pilot_final)
    V_r_final, V_i_final = bs_dnn(UE_Feedback_final)
    loss_test_Final = Lossfunc(V_r_final, V_i_final, h_final[ll], noise_std_dl)

    print(-loss_test_Final)
    experiment_result.append(-loss_test_Final.item())

🚩 Lp in 2 system, 0 epoch
✅ Model saved at epoch 0, Loss: -2.05169, Worse count reset to 0
epoch 0  anneal_param:0.5000
         loss_test:2.05169   best_test:2.05169
🚩 Lp in 2 system, 10 epoch
✅ Model saved at epoch 10, Loss: -5.52387, Worse count reset to 0
epoch 10  anneal_param:0.5000
         loss_test:5.52387   best_test:5.52387
🚩 Lp in 2 system, 20 epoch
✅ Model saved at epoch 20, Loss: -6.55258, Worse count reset to 0
epoch 20  anneal_param:0.5000
         loss_test:6.55258   best_test:6.55258
🚩 Lp in 2 system, 30 epoch
✅ Model saved at epoch 30, Loss: -7.22322, Worse count reset to 0
epoch 30  anneal_param:0.5000
         loss_test:7.22322   best_test:7.22322
🚩 Lp in 2 system, 40 epoch
✅ Model saved at epoch 40, Loss: -7.71535, Worse count reset to 0
epoch 40  anneal_param:0.5000
         loss_test:7.71535   best_test:7.71535
🚩 Lp in 2 system, 50 epoch
✅ Model saved at epoch 50, Loss: -8.20501, Worse count reset to 0
epoch 50  anneal_param:0.5000
         loss_test:8.20501   b

In [56]:
experiment_result_Lp2 = []
for ll in Lp:   
    y_pilot_final = pilot_train(hR_final[ll], hI_final[ll], K, M, L)
    UE_Feedback_final = ue_dnn(y_pilot_final)
    V_r_final, V_i_final = bs_dnn(UE_Feedback_final)
    loss_test_Final = Lossfunc(V_r_final, V_i_final, h_final[ll], noise_std_dl)

    print(-loss_test_Final)
    experiment_result_Lp2.append(-loss_test_Final.item())
print(experiment_result_Lp2)
with open('Proposed DNN trained Lp_2', 'wb') as fw:
    pickle.dump(experiment_result_Lp2, fw)

tensor(14.7376, device='cuda:0', grad_fn=<NegBackward0>)
tensor(13.2374, device='cuda:0', grad_fn=<NegBackward0>)
tensor(11.6729, device='cuda:0', grad_fn=<NegBackward0>)
tensor(10.4983, device='cuda:0', grad_fn=<NegBackward0>)
tensor(9.3969, device='cuda:0', grad_fn=<NegBackward0>)
tensor(8.6545, device='cuda:0', grad_fn=<NegBackward0>)
tensor(8.0490, device='cuda:0', grad_fn=<NegBackward0>)
[14.737640380859375, 13.23743724822998, 11.672918319702148, 10.498284339904785, 9.396851539611816, 8.654547691345215, 8.048955917358398]
